# Entraînement Flash : Visualisation avec Matplotlib & Seaborn (35 min)
## Préparation au TP de Dashboarding

Ce notebook permet de pratiquer les 4 grandes catégories de représentations graphiques vues en cours :
1. **Comparaison catégorielle** : diagramme en barres horizontales (`ax.barh`).
2. **Évolution chronologique** : courbe temporelle (`ax.plot`).
3. **Distribution & Outliers** : boîte à moustaches (`sns.boxplot`).
4. **Corrélation & Export batch** : nuage de points (`sns.scatterplot`) et sauvegarde fichier sur disque.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# Configuration du style graphique de base
sns.set_theme(style="whitegrid")
print("Bibliothèques chargées avec succès !")

### Génération du jeu de données de test
Exécutez la cellule suivante pour créer un historique des livraisons logistiques de l'enseigne.

In [ ]:
np.random.seed(42)
n = 500

dates = pd.date_range("2024-03-01", periods=31, freq="D")
regions = ["Nord", "Île-de-France", "Auvergne-Rhône-Alpes", "Nouvelle-Aquitaine", "PACA"]

df_livraisons = pd.DataFrame({
    "date": np.random.choice(dates, size=n),
    "region": np.random.choice(regions, size=n, p=[0.25, 0.35, 0.18, 0.12, 0.10]),
    "poids_colis_kg": np.random.exponential(scale=5.0, size=n).round(2) + 0.5,
    "delai_livraison_heures": np.random.normal(loc=36, scale=8, size=n).round(1),
    "frais_port_ttc": np.zeros(n)
})

# Calcul de frais réalistes corrélés au poids + bruit
df_livraisons["frais_port_ttc"] = (
    5.0 + 1.8 * df_livraisons["poids_colis_kg"] + np.random.normal(0, 1.0, size=n)
).round(2)

# Injection de quelques outliers de délai (blocages logistiques)
df_livraisons.loc[12, "delai_livraison_heures"] = 110.0
df_livraisons.loc[45, "delai_livraison_heures"] = 95.5

df_livraisons.head(3)

## Exercice 1 : Diagramme en barres horizontales (Matplotlib `fig, ax`)

**Consignes :**
1. Calculez le nombre total de livraisons par région dans une Series Pandas `stats_regions` triée par ordre croissant.
2. Créez une figure `fig1` et un axe `ax1` (`figsize=(8, 4)`).
3. Tracez un diagramme en barres horizontales (`ax1.barh`) avec :
   - En $Y$ : les noms des régions (`stats_regions.index`).
   - En $X$ : le nombre de livraisons (`stats_regions.values`).
   - Couleur des barres : `'#2b5c8f'`.
4. Ajoutez le titre `'Volume des expéditions par région'` et l'étiquette d'abscisse `'Nombre de colis'`.

In [ ]:
# TODO : Calcul et tracé horizontal
stats_regions = ...

fig1, ax1 = plt.subplots(figsize=(8, 4))
# Tracez le graphique et configurez les labels
# ...

plt.show()

In [ ]:
assert isinstance(fig1, plt.Figure), "fig1 doit être un objet matplotlib.figure.Figure."
assert len(ax1.patches) == len(regions), f"Le graphique doit comporter {len(regions)} barres horizontales."
assert ax1.get_xlabel() != "", "L'étiquette de l'axe X ne doit pas être vide."
print("✅ Exercice 1 validé !")

## Exercice 2 : Évolution chronologique d'une métrique journalière

**Consignes :**
1. Calculez le délai moyen de livraison par jour dans un DataFrame ou Series `delai_journalier` trié chronologiquement.
2. Créez une figure `fig2` et un axe `ax2` (`figsize=(10, 4)`).
3. Tracez la courbe de l'évolution du délai (`ax2.plot`) en bleu foncé (`color='teal'`), avec des marqueurs circulaires (`marker='o'`) et une épaisseur de ligne `linewidth=2`.
4. Pivotez les dates sur l'axe des abscisses à 45 degrés pour une lecture fluide (`ax2.tick_params(axis='x', rotation=45)`).
5. Ajoutez un titre explicite et le label d'axe $Y$ : `'Délai moyen (heures)'`.

In [ ]:
# TODO : Agrégation journalière et tracé temporel
delai_journalier = ...

fig2, ax2 = plt.subplots(figsize=(10, 4))
# Tracez la courbe et appliquez la rotation des dates
# ...

plt.show()

In [ ]:
assert isinstance(fig2, plt.Figure), "fig2 doit être une Figure Matplotlib."
assert len(ax2.lines) >= 1, "La courbe temporelle n'a pas été tracée sur ax2."
assert ax2.get_ylabel() != "", "Le label de l'axe Y est manquant."
print("✅ Exercice 2 validé !")

## Exercice 3 : Détection d'anomalies par région (Seaborn Boxplot)

**Consignes :**
1. Créez une figure `fig3` et un axe `ax3` (`figsize=(9, 4)`).
2. Utilisez `sns.boxplot` pour afficher la distribution des `delai_livraison_heures` par `region` en passant `ax=ax3`.
3. Titrez le graphique : `'Dispersion des délais et identification des anomalies de livraison'`.
4. Observez : quelles régions présentent des points isolés au-delà des moustaches (outliers) ?

In [ ]:
# TODO : Boxplot Seaborn orienté objet
fig3, ax3 = plt.subplots(figsize=(9, 4))
# sns.boxplot(data=..., x=..., y=..., ax=ax3)
# ...

plt.show()

In [ ]:
assert isinstance(fig3, plt.Figure), "fig3 doit être une Figure."
assert len(ax3.artists) > 0 or len(ax3.patches) > 0 or len(ax3.lines) > 0, "Aucun boxplot n'a été rendu sur ax3."
print("✅ Exercice 3 validé !")

## Exercice 4 : Nuage de points enrichi & Export batch en production

En pipeline Data, un graphique d'audit doit être sauvegardé sur disque sans bloquer la machine.

### Consignes :
1. Créez une figure `fig4` et un axe `ax4` (`figsize=(8, 5)`).
2. Utilisez `sns.scatterplot` pour croiser le `poids_colis_kg` (axe $X$) et les `frais_port_ttc` (axe $Y$), en colorant les points selon la `region` (`hue='region'`). Passez `ax=ax4`.
3. Titrez le graphique et ajoutez les labels d'axes appropriés.
4. Sauvegardez l'artefact sous `audit_frais_port.png` (`dpi=300`, `bbox_inches='tight'`).
5. Libérez la mémoire avec `plt.close(fig4)`.

In [ ]:
# TODO : Nuage de points Seaborn et export batch
fig4, ax4 = plt.subplots(figsize=(8, 5))
# sns.scatterplot(...)
# ...

# Sauvegarde et libération mémoire
fig4.savefig("audit_frais_port.png", dpi=300, bbox_inches="tight")
plt.close(fig4)

In [ ]:
assert os.path.exists("audit_frais_port.png"), "Le fichier graphique 'audit_frais_port.png' n'a pas été généré."
assert os.path.getsize("audit_frais_port.png") > 10_000, "Le fichier exporté semble anormalement petit ou vide."
print("🎉 FÉLICITATIONS ! Vous avez validé les réflexes de visualisation Matplotlib & Seaborn !")